In [5]:
DRIVE_DIR = "/content/drive/MyDrive/ns7k"
WORK = f"{DRIVE_DIR}/work"
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
%cd /content
shutil.rmtree("/content/ns7k-src", ignore_errors=True)
!git clone -q -b analysis/source-identity https://github.com/Harbani007/nonspeech7k-generalisation.git /content/ns7k-src
%cd /content/ns7k-src
!pip -q install "transformers>=4.40" soundfile
os.makedirs("/content/splits", exist_ok=True)
shutil.copy("docs/results/splits/splits.csv", "/content/splits/splits.csv")
if not os.path.exists("/content/data/audio/.unpacked"):
    for z in ["train.zip", "test.zip"]:
        zipfile.ZipFile(f"{DRIVE_DIR}/{z}").extractall("/content/data/audio")
    open("/content/data/audio/.unpacked", "w").write("ok")
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: switch runtime to GPU")

Mounted at /content/drive
/content
/content/ns7k-src
GPU: Tesla T4


In [6]:
%cd /content
import os
if not os.path.exists("/content/ns7k-src/dl_data.py"):
    !git clone -q -b analysis/source-identity https://github.com/Harbani007/nonspeech7k-generalisation.git /content/ns7k-src
%cd /content/ns7k-src
!ls dl_data.py run_gpu_cnn.sh train_cnn.py
print("WORK =", WORK if "WORK" in dir() else "NOT SET: rerun the full setup cell")

/content
/content/ns7k-src
dl_data.py  run_gpu_cnn.sh  train_cnn.py
WORK = /content/drive/MyDrive/ns7k/work


In [7]:
import os, zipfile
DRIVE_DIR = "/content/drive/MyDrive/ns7k"
print(os.listdir(DRIVE_DIR))
for z in ["train.zip", "test.zip"]:
    print("unzipping", z); zipfile.ZipFile(f"{DRIVE_DIR}/{z}").extractall("/content/data/audio")
open("/content/data/audio/.unpacked", "w").write("ok")
n = sum(f.endswith(".wav") for _, _, fs in os.walk("/content/data/audio") for f in fs)
print("wav files:", n, "(expect 7014)")
print("108160-1_0_0.wav found:", any("108160-1_0_0.wav" in fs for _, _, fs in os.walk("/content/data/audio")))

['test.zip', 'train.zip', 'ns7k_colab_bundle.zip', 'work']
unzipping train.zip
unzipping test.zip
wav files: 7014 (expect 7014)
108160-1_0_0.wav found: True


In [8]:
!python dl_data.py --audio-root /content/data/audio --splits docs/results/splits/splits.csv --out "{WORK}/dl"

reusing docs/results/splits/splits.csv 7014 clips

source_disjoint
                clips            sources          
source_disjoint  test train  val    test train val
cls                                               
breath            131  1602  117      38   126  28
cough             107   478  117      53   170  39
crying            423  1387  186      32   101  23
laugh             325   776  172      59   193  44
screaming          71   529   63      46   150  35
sneeze             52   176   38      36   117  27
yawn               51   179   34      38   124  29

official
          clips            sources          
official   test train  val    test train val
cls                                         
breath      160  1593   97      50   121  22
cough        70   538   94      55   189  33
crying      205  1601  190      38   100  18
laugh       140   959  174     117   154  27
screaming    70   523   70      70   137  24
sneeze       30   202   34      30   128  23
yawn    

In [9]:
!bash run_gpu_cnn.sh python "{WORK}/dl" "{WORK}/runs_cnn" 0 1 2
!python dl_report.py --data "{WORK}/dl" --runs "{WORK}/runs_cnn" --audit /content/splits --out "{WORK}/report_cnn" 2>&1 | grep -A14 "system"

/content/drive/MyDrive/ns7k/work/runs_cnn/cnn_source_disjoint_standard_s0: TEST (source_disjoint, best epoch 25): acc 0.754 macroF1 0.759 bal_acc 0.804; exposure {'distinct_clips_seen': 4850, 'distinct_sources_seen': 963, 'share_draws_top5_sources': 0.175, 'share_clips_top5_sources': 0.33645406670567585}
/content/drive/MyDrive/ns7k/work/runs_cnn/cnn_source_disjoint_source_s0: TEST (source_disjoint, best epoch 21): acc 0.791 macroF1 0.787 bal_acc 0.809; exposure {'distinct_clips_seen': 2966, 'distinct_sources_seen': 978, 'share_draws_top5_sources': 0.0063671875, 'share_clips_top5_sources': 0.33645406670567585}
/content/drive/MyDrive/ns7k/work/runs_cnn/cnn_source_disjoint_standard_s1: TEST (source_disjoint, best epoch 23): acc 0.750 macroF1 0.751 bal_acc 0.808; exposure {'distinct_clips_seen': 4850, 'distinct_sources_seen': 966, 'share_draws_top5_sources': 0.173671875, 'share_clips_top5_sources': 0.33645406670567585}
/content/drive/MyDrive/ns7k/work/runs_cnn/cnn_source_disjoint_source_s1